In [0]:
data = [
    (1, "Arjun Kumar", "Cardiology,Neurology", "Kochi-Kerala", "₹5,000"),
    (2, "Meera Nair", "Orthopedics", "Chennai-Tamil Nadu", "₹7,500"),
    (3, "Rahul Das", "Dermatology,Cardiology", "Bangalore-Karnataka", "₹4,200"),
    (4, "Sneha Roy", "Neurology", "Hyderabad-Telangana", "₹8,100"),
    (5, "Vikram Singh", "Orthopedics,Dermatology", "Delhi-Delhi", "₹6,000")
]
 
columns = [
    "patient_id",
    "patient_name",
    "departments",
    "location",
    "bill_amount"
]
 
df = spark.createDataFrame(data, columns)
df.show(truncate=False)
 

+----------+------------+-----------------------+-------------------+-----------+
|patient_id|patient_name|departments            |location           |bill_amount|
+----------+------------+-----------------------+-------------------+-----------+
|1         |Arjun Kumar |Cardiology,Neurology   |Kochi-Kerala       |₹5,000     |
|2         |Meera Nair  |Orthopedics            |Chennai-Tamil Nadu |₹7,500     |
|3         |Rahul Das   |Dermatology,Cardiology |Bangalore-Karnataka|₹4,200     |
|4         |Sneha Roy   |Neurology              |Hyderabad-Telangana|₹8,100     |
|5         |Vikram Singh|Orthopedics,Dermatology|Delhi-Delhi        |₹6,000     |
+----------+------------+-----------------------+-------------------+-----------+



In [0]:
from pyspark.sql.functions import *

Add first name, last name and state

In [0]:
df = df.withColumn("first_name", split(col("patient_name"), " ")[0])
df = df.withColumn("last_name", split(col("patient_name")," ")[1])
df = df.withColumn("state",split(col("location"), "-")[1])
df.show()

+----------+------------+--------------------+-------------------+-----------+----------+---------+----------+
|patient_id|patient_name|         departments|           location|bill_amount|first_name|last_name|     state|
+----------+------------+--------------------+-------------------+-----------+----------+---------+----------+
|         1| Arjun Kumar|Cardiology,Neurology|       Kochi-Kerala|     ₹5,000|     Arjun|    Kumar|    Kerala|
|         2|  Meera Nair|         Orthopedics| Chennai-Tamil Nadu|     ₹7,500|     Meera|     Nair|Tamil Nadu|
|         3|   Rahul Das|Dermatology,Cardi...|Bangalore-Karnataka|     ₹4,200|     Rahul|      Das| Karnataka|
|         4|   Sneha Roy|           Neurology|Hyderabad-Telangana|     ₹8,100|     Sneha|      Roy| Telangana|
|         5|Vikram Singh|Orthopedics,Derma...|        Delhi-Delhi|     ₹6,000|    Vikram|    Singh|     Delhi|
+----------+------------+--------------------+-------------------+-----------+----------+---------+----------+



Rewrite location to city

In [0]:
df = df.withColumn("location", split(col("location"),"-")[0])
df.show()

+----------+------------+--------------------+---------+-----------+----------+---------+----------+
|patient_id|patient_name|         departments| location|bill_amount|first_name|last_name|     state|
+----------+------------+--------------------+---------+-----------+----------+---------+----------+
|         1| Arjun Kumar|Cardiology,Neurology|    Kochi|     ₹5,000|     Arjun|    Kumar|    Kerala|
|         2|  Meera Nair|         Orthopedics|  Chennai|     ₹7,500|     Meera|     Nair|Tamil Nadu|
|         3|   Rahul Das|Dermatology,Cardi...|Bangalore|     ₹4,200|     Rahul|      Das| Karnataka|
|         4|   Sneha Roy|           Neurology|Hyderabad|     ₹8,100|     Sneha|      Roy| Telangana|
|         5|Vikram Singh|Orthopedics,Derma...|    Delhi|     ₹6,000|    Vikram|    Singh|     Delhi|
+----------+------------+--------------------+---------+-----------+----------+---------+----------+



create address location concatinating location and state

In [0]:
df = df.withColumn("address_location", concat_ws(", ", col("location"),col("state")))
df.show()

+----------+------------+--------------------+---------+-----------+----------+---------+----------+--------------------+
|patient_id|patient_name|         departments| location|bill_amount|first_name|last_name|     state|    address_location|
+----------+------------+--------------------+---------+-----------+----------+---------+----------+--------------------+
|         1| Arjun Kumar|Cardiology,Neurology|    Kochi|     ₹5,000|     Arjun|    Kumar|    Kerala|       Kochi, Kerala|
|         2|  Meera Nair|         Orthopedics|  Chennai|     ₹7,500|     Meera|     Nair|Tamil Nadu| Chennai, Tamil Nadu|
|         3|   Rahul Das|Dermatology,Cardi...|Bangalore|     ₹4,200|     Rahul|      Das| Karnataka|Bangalore, Karnataka|
|         4|   Sneha Roy|           Neurology|Hyderabad|     ₹8,100|     Sneha|      Roy| Telangana|Hyderabad, Telangana|
|         5|Vikram Singh|Orthopedics,Derma...|    Delhi|     ₹6,000|    Vikram|    Singh|     Delhi|        Delhi, Delhi|
+----------+------------

create patient info concatinating patient name and department

In [0]:
df.withColumn("patient_info", concat_ws("-", df["patient_name"], df["departments"])).show()

+----------+------------+--------------------+---------+-----------+----------+---------+----------+--------------------+--------------------+----------+
|patient_id|patient_name|         departments| location|bill_amount|first_name|last_name|     state|    address_location|        patient_info|state_code|
+----------+------------+--------------------+---------+-----------+----------+---------+----------+--------------------+--------------------+----------+
|         1| Arjun Kumar|Cardiology,Neurology|    Kochi|     ₹5,000|     Arjun|    Kumar|    Kerala|       Kochi, Kerala|Arjun Kumar-Cardi...|        Ke|
|         2|  Meera Nair|         Orthopedics|  Chennai|     ₹7,500|     Meera|     Nair|Tamil Nadu| Chennai, Tamil Nadu|Meera Nair-Orthop...|        Ta|
|         3|   Rahul Das|Dermatology,Cardi...|Bangalore|     ₹4,200|     Rahul|      Das| Karnataka|Bangalore, Karnataka|Rahul Das-Dermato...|        Ka|
|         4|   Sneha Roy|           Neurology|Hyderabad|     ₹8,100|     Sne

In [0]:
df = df.drop("patient_info")

create a state code column using 1st 2letters of the state


In [0]:
df = df.withColumn("state_code", substring(col("state"), 1, 2))
df.show()

+----------+------------+--------------------+---------+-----------+----------+---------+----------+--------------------+----------+
|patient_id|patient_name|         departments| location|bill_amount|first_name|last_name|     state|    address_location|state_code|
+----------+------------+--------------------+---------+-----------+----------+---------+----------+--------------------+----------+
|         1| Arjun Kumar|Cardiology,Neurology|    Kochi|     ₹5,000|     Arjun|    Kumar|    Kerala|       Kochi, Kerala|        Ke|
|         2|  Meera Nair|         Orthopedics|  Chennai|     ₹7,500|     Meera|     Nair|Tamil Nadu| Chennai, Tamil Nadu|        Ta|
|         3|   Rahul Das|Dermatology,Cardi...|Bangalore|     ₹4,200|     Rahul|      Das| Karnataka|Bangalore, Karnataka|        Ka|
|         4|   Sneha Roy|           Neurology|Hyderabad|     ₹8,100|     Sneha|      Roy| Telangana|Hyderabad, Telangana|        Te|
|         5|Vikram Singh|Orthopedics,Derma...|    Delhi|     ₹6,000| 

clean up the bill amount and cast to integer

In [0]:
df = df.withColumn("bill_amount", regexp_replace(col("bill_amount"),"[₹,]","").cast("int"))
df.show()
df.printSchema()

+----------+------------+--------------------+---------+-----------+----------+---------+----------+--------------------+----------+
|patient_id|patient_name|         departments| location|bill_amount|first_name|last_name|     state|    address_location|state_code|
+----------+------------+--------------------+---------+-----------+----------+---------+----------+--------------------+----------+
|         1| Arjun Kumar|Cardiology,Neurology|    Kochi|       5000|     Arjun|    Kumar|    Kerala|       Kochi, Kerala|        Ke|
|         2|  Meera Nair|         Orthopedics|  Chennai|       7500|     Meera|     Nair|Tamil Nadu| Chennai, Tamil Nadu|        Ta|
|         3|   Rahul Das|Dermatology,Cardi...|Bangalore|       4200|     Rahul|      Das| Karnataka|Bangalore, Karnataka|        Ka|
|         4|   Sneha Roy|           Neurology|Hyderabad|       8100|     Sneha|      Roy| Telangana|Hyderabad, Telangana|        Te|
|         5|Vikram Singh|Orthopedics,Derma...|    Delhi|       6000| 

create a patient code with 1st 2 letters of first name and last name along with patient id.

In [0]:
df = df.withColumn("patient_code", concat_ws("", substring(col("first_name"),1,2),substring(col("last_name"),1,2), col("patient_id")))
df.show()

+----------+------------+--------------------+---------+-----------+----------+---------+----------+--------------------+----------+------------+
|patient_id|patient_name|         departments| location|bill_amount|first_name|last_name|     state|    address_location|state_code|patient_code|
+----------+------------+--------------------+---------+-----------+----------+---------+----------+--------------------+----------+------------+
|         1| Arjun Kumar|Cardiology,Neurology|    Kochi|       5000|     Arjun|    Kumar|    Kerala|       Kochi, Kerala|        Ke|       ArKu1|
|         2|  Meera Nair|         Orthopedics|  Chennai|       7500|     Meera|     Nair|Tamil Nadu| Chennai, Tamil Nadu|        Ta|       MeNa2|
|         3|   Rahul Das|Dermatology,Cardi...|Bangalore|       4200|     Rahul|      Das| Karnataka|Bangalore, Karnataka|        Ka|       RaDa3|
|         4|   Sneha Roy|           Neurology|Hyderabad|       8100|     Sneha|      Roy| Telangana|Hyderabad, Telangana|   